In [1]:
import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import seaborn as sns
import torch
import tensorflow_hub as hub
from gensim.models import Word2Vec, FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel


2025-12-30 16:51:21.544202: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/codespace/.python/current/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/home/codespace/.python/current/lib/python3.12/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/home/codespace/.python/current/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: I

In [2]:
sentences = [
    "I love playing football on weekends.",
    "Football players train every day.",
    "Machine learning is fun and powerful.",
    "I enjoy working with neural networks and embeddings.",
    "The weather today is sunny and warm.",
    "Cats and dogs are common pets.",
    "I went to the bank to deposit money.",
    "The river bank was full of flowers."
]
tokenized = [s.lower().split() for s in sentences]

In [3]:
print("\n=== TASK 1: Word2Vec Skip-Gram ===")
w2v = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, sg=1, epochs=20)
word = "football"
if word in w2v.wv:
    print(w2v.wv.most_similar(word, topn=10))


=== TASK 1: Word2Vec Skip-Gram ===
[('players', 0.3508923649787903), ('working', 0.30288901925086975), ('neural', 0.17957563698291779), ('networks', 0.16514906287193298), ('common', 0.16426755487918854), ('i', 0.1514052450656891), ('day.', 0.11366388201713562), ('cats', 0.11242301017045975), ('sunny', 0.09591232240200043), ('love', 0.08172277361154556)]


In [4]:
def cos_sim(a, b): return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-9))
pairs = [("king","queen"), ("cat","dog"), ("football","soccer"), ("car","bicycle")]

for a, b in pairs:
    if a in w2v.wv and b in w2v.wv:
        print(f"{a} - {b}: {cos_sim(w2v.wv[a], w2v.wv[b]):.4f}")
    else:
        print(f"{a} or {b} not in vocabulary.")

king or queen not in vocabulary.
cat or dog not in vocabulary.
football or soccer not in vocabulary.
car or bicycle not in vocabulary.


In [5]:
ft = FastText(sentences=tokenized, vector_size=100, window=5, min_count=1, epochs=20)
oov_words = ["footbal", "footballer", "neuralnetwork", "unknownword"]
for w in oov_words:
    print(f"fastText('{w}') similarity to 'football':", ft.wv.similarity('football', w))
    try:
        print(f"word2vec('{w}') similarity to 'football':", w2v.wv.similarity('football', w))
    except:
        print(f"'{w}' not in Word2Vec vocab")

fastText('footbal') similarity to 'football': 0.7158922
'footbal' not in Word2Vec vocab
fastText('footballer') similarity to 'football': 0.68178594
'footballer' not in Word2Vec vocab
fastText('neuralnetwork') similarity to 'football': -0.3016209
'neuralnetwork' not in Word2Vec vocab
fastText('unknownword') similarity to 'football': -0.05591111
'unknownword' not in Word2Vec vocab


In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
# Gensim w2v model needs to be loaded/trained here as well if it isn't already 'w2v'

texts = ["I love this product", "This is terrible", "I enjoyed the movie", "I hate the food"]
labels = [1, 0, 1, 0]  # 1=positive, 0=negative

# Assuming 'w2v' is a loaded gensim Word2Vec model, otherwise this function will also fail
def sent_vec(sent):
    tokens = sent.lower().split()
    # This part assumes a model named 'w2v' with vector size 100 exists and is trained/loaded
    vecs = [w2v.wv[w] for w in tokens if w in w2v.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

# Now you can run the rest of your original code in the same cell or a subsequent cell:
X = np.vstack([sent_vec(s) for s in texts])
y = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
)
clf = LogisticRegression(max_iter=500).fit(X_train, y_train)
pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.0
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       1.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



In [7]:
import tensorflow as tf
import tensorflow_hub as hub # assuming this is already imported from your previous session

elmo = hub.load("https://tfhub.dev/google/elmo/3")
sentences_elmo = ["I love the bank of the river.", "He went to the bank to deposit money."]
embs = elmo.signatures["default"](tf.constant(sentences_elmo))["elmo"].numpy()
print("ELMo embeddings shape:", embs.shape)

2025-12-30 16:51:46.424960: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


ELMo embeddings shape: (2, 8, 1024)


In [8]:
idx0 = sentences_elmo[0].split().index("bank")
idx1 = sentences_elmo[1].split().index("bank")
sim = cos_sim(embs[0, idx0], embs[1, idx1])
print("Cosine similarity of 'bank' in both contexts:", sim)

Cosine similarity of 'bank' in both contexts: 0.6496587991714478


In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

s1 = "I went to the bank to withdraw money."
s2 = "The boat was near the bank of the river."

inputs1 = tokenizer(s1, return_tensors="pt")
inputs2 = tokenizer(s2, return_tensors="pt")

with torch.no_grad():
    out1 = model(**inputs1).last_hidden_state.mean(1)
    out2 = model(**inputs2).last_hidden_state.mean(1)

print("Sentence similarity (BERT mean pooling):", cos_sim(out1[0].numpy(), out2[0].numpy()))

Sentence similarity (BERT mean pooling): 0.6070259213447571


In [10]:
docs = [TaggedDocument(words=s.lower().split(), tags=[f"D{i}"]) for i, s in enumerate(sentences)]
d2v = Doc2Vec(docs, vector_size=50, window=5, min_count=1, epochs=40)
new_doc = "I enjoy playing football at the stadium."
inferred = d2v.infer_vector(new_doc.lower().split())
print("Most similar documents:", d2v.dv.most_similar([inferred], topn=3))

Most similar documents: [('D3', 0.3267967402935028), ('D7', 0.31555384397506714), ('D0', 0.2796604633331299)]


In [11]:
# Change this line to a valid model name
model_st = SentenceTransformer('all-distilroberta-v1') 

train_examples = [
    InputExample(texts=["A man is playing a guitar.", "A person plays a guitar."], label=0.9),
    InputExample(texts=["A dog is running.", "A cat is sleeping."], label=0.0)
]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)
train_loss = losses.CosineSimilarityLoss(model_st)
model_st.fit(train_objectives=[(train_dataloader, train_loss)], epochs=1)

emb1 = model_st.encode("A man plays guitar.")
emb2 = model_st.encode("A person plays the guitar.")
print("Cosine similarity (RoBERTa):", util.cos_sim(emb1, emb2))

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

In [ ]:
def benchmark(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    mod = AutoModel.from_pretrained(model_name)
    texts = ["This is a test sentence."] * 8
    inputs = tok(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        _ = mod(**inputs)
    import time
    t0 = time.time()
    with torch.no_grad():
        _ = mod(**inputs)
    t = time.time() - t0
    n_params = sum(p.numel() for p in mod.parameters())
    return model_name, n_params, t

b = benchmark("bert-base-uncased")
d = benchmark("distilbert-base-uncased")
print(f"{b[0]}: params={b[1]/1e6:.1f}M, time={b[2]:.3f}s")
print(f"{d[0]}: params={d[1]/1e6:.1f}M, time={d[2]:.3f}s")

In [ ]:
words = list(w2v.wv.index_to_key)[:50]
vecs = [w2v.wv[w] for w in words]
pca = PCA(n_components=2).fit_transform(vecs)
plt.figure(figsize=(10,8))
plt.scatter(pca[:,0], pca[:,1])
for i, w in enumerate(words):
    plt.annotate(w, (pca[i,0], pca[i,1]))
plt.title("PCA of Word2Vec Embeddings")
plt.show()